# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** The handover document: what an editor does on Monday
morning, what they must check first, what this system must never be allowed to decide, and what would tell
us it has gone stale.

Continues from `w05_model.ipynb` (ML-08, the shipped ranking) and `w06_validation_audit.ipynb` (ML-09, the
audit). **Every rule in this playbook exists because something was measured** — the ML-08 error analysis
found four specific failure modes, and each one becomes an operational guardrail below rather than a
caveat nobody reads.

This notebook writes the artifact the paper points at: `work/outputs/refresh_queue.csv`.

In [1]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import json
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from pandas.api.types import is_numeric_dtype

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
groups = df["client_id"].to_numpy()
BASE_RATE = float(y.mean())
CAPACITY_PER_SPRINT = 50


def precision_at_k(labels, scores, k: int) -> float:
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


def roc_auc(labels, scores) -> float:
    labels = np.asarray(labels)
    n_pos, n_neg = labels.sum(), (1 - labels).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = pd.Series(np.asarray(scores, dtype=float)).rank().to_numpy()
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


receipt = json.loads(Path("work/outputs/model_metrics.json").read_text())
print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows | {df['client_id'].nunique()} clients | base rate {BASE_RATE:.4f}")
print(f"ML-08 receipt: shipped = {receipt['shipped']}, p@50 = "
      f"{receipt['systems'][receipt['shipped']]['p@50']}")

Working dir: C:\Users\real time\Desktop\Rayan_flyrank
Loaded 30,000 rows | 32 clients | base rate 0.5421
ML-08 receipt: shipped = hybrid_rule_band_plus_logistic, p@50 = 0.9


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The ranking.** The ML-07 rule selects the band and supplies the reason codes; the ML-08 logistic model
orders pages inside it. Out-of-fold precision@50 = **0.900** against a **0.542** base rate — 45 of 50 slots
on pages measured as declining, versus 27 for random triage.

**Why reason codes and not a score.** An editor cannot act on "0.94". They can act on *"this page has
sustained coverage, real demand, sits mid-page-one, hasn't been touched in a quarter, and is old enough to
have settled"*. Every row in the queue carries the codes that fired, in plain words:

| Reason code | What it means in English | Points |
|---|---|---|
| `established_coverage` | Shows up in search on 20–87 of the last 88 days — real, sustained visibility | 3 |
| `has_demand` | At least 40 impressions in 90 days — enough that a refresh could matter | 2 |
| `mid_position` | Ranks between 4 and 50 — findable, but not already winning | 2 |
| `stale_90d` | Not updated in at least a quarter | 1 |
| `mature_page` | Between 90 and 364 days old — settled, still current | 1 |

**All 50 rows of the sprint queue carry all five codes.** The rule alone cannot separate them; the model
decides the order within that band. That division of labour is the whole design: **the part a human has to
trust is the part a human can read, and the part a human cannot read only reorders it.**

### The action table

| Situation | Action | Confidence | What would make it wrong |
|---|---|---|---|
| All five codes, `avg_position` ≥ 11 | **Refresh review** — update facts, realign headers with current intent, expand thin sections | **Medium-high.** 0.900 measured at K=50 | Off-season intent looks identical; no seasonality field exists to rule it out |
| All five codes, **`avg_position` < 11 (page one)** | **Refresh review — confirm direction first** | **Medium.** Every measured error in the top 50 lives here | 4 of 5 top-50 misses were page-1 pages that had *grown* +23.6% to +93.7% |
| `content_type == comparison article` | **Remove from the queue** | **None.** Out-of-fold AUC **0.524** = chance | 697 pages the model cannot order at all. Neither can the rule |
| Client with fewer than ~200 scored pages | **Do not use the ranking** | **Unknown, not low** | Per-client AUC is only measurable at scale; below that it is unmeasured, not safe |
| Beyond the per-client cap | **Defer to next sprint** | — | Not a quality judgement — a portfolio-coverage rule |
| `no_signal` (no codes fired) | **Leave alone** | — | 75 pages, decline rate 0.293 — below the base rate |

### The three operational rules, and what each one costs

None of these are model improvements. They are product decisions layered on top, and the cell below
reports the queue's precision before and after so the two can never be confused.

1. **Cap at 8 pages per client.** The unfiltered top 50 spans 7 of 32 clients with one supplying **58%**
   (29 of 50 slots) — *worse* portfolio coverage than the ML-07 rule's 44%. Better ranking, more
   concentration. An editor serving the whole book cannot spend 58% of a sprint on one portfolio.
2. **Flag page-1 rows for a direction check.** Cheap: one glance at the trend line before assigning.
3. **Drop `comparison article`.** The model is provably at chance on them; ranking them is false precision.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.base import clone

# --- the ML-07 rule: the band and the reason codes ------------------------
position = df["avg_position"].replace(0, np.nan)
RULE = {
    "established_coverage": (df["days_with_impressions"].between(20, 87), 3),
    "has_demand":           (df["impressions_90d"] >= 40, 2),
    "mid_position":         (((position > 3) & (position <= 50)).fillna(False), 2),
    "stale_90d":            (df["days_since_last_update"] >= 90, 1),
    "mature_page":          (df["content_age_days"].between(90, 364), 1),
}
band = sum(flag.astype(int) * pts for flag, pts in RULE.values()).to_numpy()
flag_frame = pd.DataFrame({name: flag.to_numpy() for name, (flag, _) in RULE.items()})
reason_codes = [",".join(flag_frame.columns[row]) or "no_signal" for row in flag_frame.to_numpy()]

# --- the ML-08 model: the ordering inside the band ------------------------
FEATURES = [
    "search_volume", "competition", "competition_level", "cpc",
    "word_count", "char_count", "word_count_tier", "char_count_tier",
    "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "age_tier", "age_tier_order", "days_since_last_update", "freshness_tier",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "impression_tier", "position_tier",
]
X = df[FEATURES].copy()
X["avg_position"] = X["avg_position"].replace(0, np.nan)
for c in ["search_volume", "competition", "cpc", "word_count", "char_count", "avg_position"]:
    X[f"has_{c}"] = X[c].notna().astype(int)
COUNT_COLS = ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
              "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "search_volume"]
X[COUNT_COLS] = np.log1p(X[COUNT_COLS])
CAT = [c for c in FEATURES if not is_numeric_dtype(df[c])]
NUM = [c for c in X.columns if c not in CAT]

FORBIDDEN = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
             "is_declining_label", "content_id", "client_id"}
assert not FORBIDDEN & set(X.columns), "a forbidden column reached the queue"

pre = ColumnTransformer([
    ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), NUM),
    ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="__missing__")),
                      ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), CAT),
])
oof = np.full(len(df), np.nan)
for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
    assert not (set(groups[tr]) & set(groups[te])), "client leaked across the split"
    pipe = Pipeline([("pre", clone(pre)),
                     ("clf", LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE))])
    oof[te] = pipe.fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
assert not np.isnan(oof).any(), "a row was never scored"

hybrid = band * 1000 + oof * 100
assert round(precision_at_k(y, hybrid, 50), 4) == receipt["systems"][receipt["shipped"]]["p@50"], \
    "this notebook disagrees with the committed ML-08 receipt"
print(f"ranking reproduced and matches the ML-08 receipt: p@50 = {precision_at_k(y, hybrid, 50):.3f}\n")

# --- build the queue -------------------------------------------------------
queue = df[["content_id", "client_id", "content_type", "impressions_90d", "days_with_impressions",
            "avg_position", "days_since_last_update", "content_age_days", "is_declining_label"]].copy()
queue["model_score"] = oof
queue["baseline_score"] = band
queue["reason_codes"] = reason_codes
queue["hybrid_score"] = hybrid
queue = queue.sort_values("hybrid_score", ascending=False, kind="stable").reset_index(drop=True)
queue.insert(0, "queue_rank", np.arange(1, len(queue) + 1))

# --- the operational rules, applied in order, each one costed --------------
PER_CLIENT_CAP = 8
MIN_CLIENT_PAGES = 200
client_sizes = df["client_id"].value_counts()

sprint = queue.head(CAPACITY_PER_SPRINT).copy()
sprint["action"] = "refresh_review"
sprint.loc[sprint["avg_position"] < 11, "action"] = "refresh_review_verify_direction"
sprint.loc[sprint["content_type"] == "comparison article", "action"] = "removed_model_at_chance"
sprint.loc[sprint["client_id"].map(client_sizes) < MIN_CLIENT_PAGES, "action"] = "removed_client_too_small"
sprint["rank_in_client"] = sprint.groupby("client_id").cumcount() + 1
sprint.loc[(sprint["rank_in_client"] > PER_CLIENT_CAP) &
           (sprint["action"].str.startswith("refresh")), "action"] = "deferred_client_cap"

print("Sprint queue after the operational rules:")
print(sprint["action"].value_counts().to_string())

sendable = sprint[sprint["action"].str.startswith("refresh_review")]
print(f"\nsent to an editor this sprint: {len(sendable)} of {CAPACITY_PER_SPRINT}")
print(f"  measured declining in that set : {sendable['is_declining_label'].mean():.3f} "
      f"(base rate {BASE_RATE:.3f})")
print(f"  needing a direction check first: "
      f"{(sendable['action'] == 'refresh_review_verify_direction').sum()}")
print(f"  clients represented            : {sendable['client_id'].nunique()} "
      f"(unfiltered top 50: {sprint['client_id'].nunique()})")
print(f"  largest single client's share  : "
      f"{sendable['client_id'].value_counts().iloc[0] / len(sendable):.0%} "
      f"(unfiltered: {sprint['client_id'].value_counts().iloc[0] / len(sprint):.0%})")

print("\nThe two numbers, kept apart on purpose:")
print(f"  model precision@50 (the honest headline)     {precision_at_k(y, hybrid, 50):.3f}")
print(f"  precision of the filtered sendable set       {sendable['is_declining_label'].mean():.3f}")
print("  The filters trade slots for portfolio coverage and safety. That is a HUMAN decision")
print("  layered on the ranking - it must never be reported as model performance.")

print("\nReason codes on the sendable set (counts only, no identifiers):")
print(sendable["reason_codes"].value_counts().to_string())

ranking reproduced and matches the ML-08 receipt: p@50 = 0.900

Sprint queue after the operational rules:
action
deferred_client_cap                23
refresh_review_verify_direction    22
refresh_review                      5

sent to an editor this sprint: 27 of 50
  measured declining in that set : 0.926 (base rate 0.542)
  needing a direction check first: 22
  clients represented            : 7 (unfiltered top 50: 7)
  largest single client's share  : 30% (unfiltered: 58%)

The two numbers, kept apart on purpose:
  model precision@50 (the honest headline)     0.900
  precision of the filtered sendable set       0.926
  The filters trade slots for portfolio coverage and safety. That is a HUMAN decision
  layered on the ranking - it must never be reported as model performance.

Reason codes on the sendable set (counts only, no identifiers):
reason_codes
established_coverage,has_demand,mid_position,stale_90d,mature_page    27


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

**Who.** A content strategist or SEO lead planning a sprint's editorial capacity across a portfolio they
already own and understand.

**For what.** One question only: *of the pages we could open this sprint, which ones first?* The output is
a **reading order**, not a verdict on any page.

**How it enters a workflow.** It replaces the triage step — "anything not touched in six months", a rule
that in this dataset fires on **174 pages of 30,000** and has essentially nothing to rank. It does not
replace the editor's judgement about what to *do* with a page once opened.

### Where it stops being valid

| Boundary | Why | Evidence |
|---|---|---|
| **Only mature, still-trafficked pages** | The file was pre-filtered upstream: min age exactly 90 days, every row has traffic > 0 | Pages that died completely are absent — this queue cannot find them |
| **Only this portfolio's shape** | Per-client out-of-fold AUC 0.506–0.770 | For some clients the ranking is barely better than random |
| **Only concurrent, never predictive** | Label and features share the same 90-day window; the file has no dates | "Resembles pages measured as declining", never "will decline" |
| **Only impressions** | The label reads impression change, nothing else | A page holding impressions while losing clicks is invisible to it |
| **Only at small K** | The ML-07 rule beats it at K=200 | If capacity grows past ~100 pages/sprint, re-run the comparison |
| **Not on new clients** | A model that never saw a client ranks it worst | Onboarding a client means no queue until there is history |

### The four things this is not

1. **Not a decline detector.** Its top feature is impressions volume by 2.4× over the next. It ranks
   substantially by *exposure*, and a page can be highly exposed and perfectly healthy — which is exactly
   how its measured errors happen.
2. **Not a forecast.** No date column, no future window, no intervention.
3. **Not a content-quality score.** It reads traffic shape. It has never seen a sentence of the page.
4. **Not a measure of anyone's work.** It cannot distinguish a page decaying because the writer was
   careless from one decaying because the topic died.

### The claim, in one sentence

> On a client-held-out split of 30,000 mature, still-trafficked pages, this ranking placed 45 of its top
> 50 on items measured as declining (precision@50 = 0.900 vs a 0.542 base rate). It is decision-support
> for triage in this portfolio and window. It does not establish that these pages will keep declining, and
> nothing here supports a claim that refreshing them recovers traffic.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
print("VALIDITY BOUNDARIES, each one measured rather than asserted")

print(f"\n1. population it was fitted on:")
print(f"   min content_age_days {df['content_age_days'].min()} | rows with zero 90d impressions "
      f"{(df['impressions_90d'] == 0).sum()} | zero sessions {(df['sessions_90d'] == 0).sum()}")
print("   -> mature, still-trafficked pages only. Dead pages are not in the population.")

print(f"\n2. per-client reliability (out-of-fold AUC, clients with >=200 pages):")
pc = pd.DataFrame([(len(g), roc_auc(y[g.index], oof[g.index]))
                   for _, g in df.groupby("client_id") if len(g) >= MIN_CLIENT_PAGES],
                  columns=["pages", "auc"]).dropna().sort_values("auc")
print(f"   {len(pc)} clients | AUC {pc['auc'].min():.3f} to {pc['auc'].max():.3f} "
      f"| median {pc['auc'].median():.3f}")
print(f"   clients where the ranking is near chance (AUC < 0.55): {(pc['auc'] < 0.55).sum()}")
print(f"   clients below the {MIN_CLIENT_PAGES}-page floor, where AUC is unmeasurable: "
      f"{(client_sizes < MIN_CLIENT_PAGES).sum()} of {len(client_sizes)}")

print(f"\n3. by content type - where it must not be used:")
for t, g in df.groupby("content_type"):
    idx = g.index.to_numpy()
    auc = roc_auc(y[idx], oof[idx])
    verdict = "AT CHANCE - remove from queue" if auc < 0.56 else "usable"
    print(f"   {t:<20} n={len(idx):>6,}  AUC {auc:.3f}  {verdict}")

print(f"\n4. the label reads impressions ONLY - a clicks-based label would disagree:")
clicks_down = (df["clicks_last_30d"] < df["clicks_prev_30d"]).astype(int).to_numpy()
agree = (clicks_down == y).mean()
print(f"   agreement between the impressions label and a clicks-direction label: {agree:.3f}")
print(f"   -> {1 - agree:.1%} of pages would be classified differently by a clicks-based definition.")

print(f"\n5. where the rule still beats the model (capacity matters):")
sysm = receipt["systems"]
for k in ("p@20", "p@50", "p@100", "p@200", "p@500"):
    r, s = sysm["baseline_rule"][k], sysm[receipt["shipped"]][k]
    print(f"   {k:<6} rule {r:.3f} | shipped {s:.3f} | {'RULE WINS' if r > s else 'shipped wins'}")
print("   -> the shipped ranking is only the right choice while capacity stays around 50-100 pages.")

VALIDITY BOUNDARIES, each one measured rather than asserted

1. population it was fitted on:
   min content_age_days 90 | rows with zero 90d impressions 0 | zero sessions 0
   -> mature, still-trafficked pages only. Dead pages are not in the population.

2. per-client reliability (out-of-fold AUC, clients with >=200 pages):
   21 clients | AUC 0.506 to 0.770 | median 0.642
   clients where the ranking is near chance (AUC < 0.55): 2
   clients below the 200-page floor, where AUC is unmeasurable: 10 of 32

3. by content type - where it must not be used:


   comparison article   n=   697  AUC 0.524  AT CHANCE - remove from queue
   feedly article       n= 2,096  AUC 0.843  usable
   keyword article      n=27,207  AUC 0.660  usable

4. the label reads impressions ONLY - a clicks-based label would disagree:
   agreement between the impressions label and a clicks-direction label: 0.534
   -> 46.6% of pages would be classified differently by a clicks-based definition.

5. where the rule still beats the model (capacity matters):
   p@20   rule 0.750 | shipped 0.950 | shipped wins
   p@50   rule 0.740 | shipped 0.900 | shipped wins
   p@100  rule 0.800 | shipped 0.850 | shipped wins
   p@200  rule 0.835 | shipped 0.805 | RULE WINS
   p@500  rule 0.808 | shipped 0.770 | RULE WINS
   -> the shipped ranking is only the right choice while capacity stays around 50-100 pages.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### The pre-flight check — three questions, about two minutes per page

Every page in the queue gets these before an editor spends an hour on it:

1. **Is it actually going down?** Open the 90-day impression trend. **This is not optional for page-1
   rows** — all five of the shipped queue's top-50 misses sat at `avg_position` 4.1–10.5, and four of them
   were *growing* by +23.6% to +93.7%. The model ranks by exposure and cannot tell a healthy page-one page
   from a decaying one.
2. **Is the decline seasonal or intent-driven?** A page about a seasonal topic measured out of season looks
   identical to a decaying one. **The dataset has no seasonality field, so this failure mode is
   unquantified — I cannot tell you how often it happens, only that nothing rules it out.** A human who
   knows the content calendar can answer it in seconds.
3. **Is a refresh even the lever?** If the topic is dead, the SERP intent has changed, or the page was
   deliberately deprioritised, the right action is retire or consolidate — not refresh. The model has never
   read the page.

### The no-go list — what this system must never be allowed to do

| Never | Why |
|---|---|
| **Auto-publish or auto-edit any page** | The system recommends *opening* a page. It has never read one, and it has no measure of editorial quality |
| **Auto-delete, deindex, or unpublish** | A low score means "not a refresh priority", not "worthless". Nothing here measures a page's strategic or commercial value |
| **Rank `comparison article`** | Measured at chance (AUC 0.524). Ranking them produces false precision with a confident-looking number attached |
| **Run on a client with no history** | The measured cost of an unseen client is 0.152 AUC. A new client's queue would be the model's weakest output presented identically to its strongest |
| **Be reported to a client as performance measurement** | It is a triage aid on a proxy label, not an account health metric. The label is a −20% threshold someone chose |
| **Be used to evaluate a writer or team** | It cannot separate "written badly" from "topic died". Using it this way would be measuring the wrong thing at someone's expense |
| **Be presented as a forecast** | Concurrent detection only. "Will decline" is not a sentence this data can support |
| **Run without the base rate beside it** | 0.900 means nothing without 0.542. Any dashboard showing one must show the other |
| **Have `client_id` or `content_id` added as features** | Pseudonymous IDs; a model that learns an ID has learned nothing transferable |
| **Be retrained on a label built from the model's own picks** | The decision log records human choices *influenced by this queue*. Training on it closes a feedback loop and relabels the world in the model's own image |

### The escalation rule

**If an editor overrules the queue three times in one sprint for the same reason, that is a bug report,
not a disagreement.** Log the reason and re-open the error analysis. The `comparison article` exclusion in
this playbook is precisely what that process produces when it works.

### The one thing a human must do that the system cannot

**Log every decision — refreshed, skipped, or monitored — with a date and a one-line reason.** The dataset
has no record of any intervention, which is why nothing in this project can make a causal claim. That log
is the missing outcome column. After two quarters it would support the question this system genuinely
cannot answer today: *does refreshing the pages we pick actually recover traffic?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
print("THE PRE-FLIGHT CHECK, justified by the measured error pattern")
order = np.argsort(-hybrid, kind="stable")
top50 = df.iloc[order[:CAPACITY_PER_SPRINT]]
misses = top50[top50["is_declining_label"] == 0]
print(f"\ntop-{CAPACITY_PER_SPRINT} misses: {len(misses)} of {CAPACITY_PER_SPRINT}")
print(misses[["impressions_90d", "days_with_impressions", "avg_position",
              "content_type", "trend_direction", "trend_pct"]].round(2).to_string(index=False))
print(f"  every miss is a page-1 page: avg_position "
      f"{misses['avg_position'].min():.1f}-{misses['avg_position'].max():.1f}")
print(f"  measured 'up': {(misses['trend_direction'] == 'up').sum()} of {len(misses)}")
print("  -> CHECK 1 (confirm the direction on page-1 rows) is not caution, it is where the errors are.")

# How much of the sprint needs the check?
need_check = (top50["avg_position"] < 11).sum()
print(f"\n  pages in the top {CAPACITY_PER_SPRINT} needing the direction check: {need_check} "
      f"({need_check / CAPACITY_PER_SPRINT:.0%} of the sprint)")

print("\nCHECK 2 (seasonality) is UNQUANTIFIABLE from this file, stated rather than hidden:")
# Match whole name-tokens, not substrings: "days_since_last_update" contains "date" as a substring
# and would otherwise register as a calendar field. (Same trap ML-04 flagged.)
import re
SEASON_TOKENS = {"season", "seasonal", "month", "quarter", "date", "calendar", "week", "year"}
found = [c for c in df.columns if SEASON_TOKENS & set(re.split(r"_+", c.lower()))]
print(f"  columns that could identify seasonality or calendar position: {found or 'none'}")
assert not found, "a seasonality field exists - this failure mode could be quantified after all"
print("  -> I cannot report how often seasonality causes a false positive. Only that nothing excludes it.")

print("\nTHE NO-GO LIST, enforced in code where it can be:")
print(f"  IDs as features            : blocked by assertion (FORBIDDEN set) -> "
      f"{sorted(FORBIDDEN & set(X.columns)) or 'clean'}")
at_chance = [t for t, g in df.groupby("content_type") if roc_auc(y[g.index], oof[g.index]) < 0.56]
print(f"  content types at chance    : {at_chance} -> removed from the queue above")
small = client_sizes[client_sizes < MIN_CLIENT_PAGES]
print(f"  clients below the {MIN_CLIENT_PAGES}-page floor: {len(small)} clients, "
      f"{small.sum():,} pages -> ranking withheld")
print(f"  base rate travels with every metric: {BASE_RATE:.4f} (written into the exported queue's header)")

print("\nWHAT THE HUMAN MUST SUPPLY - the column this dataset does not have:")
OUTCOME_HINTS = ("refresh", "action_taken", "reviewed", "outcome", "intervention")
print(f"  intervention/outcome columns present: "
      f"{[c for c in df.columns if any(h in c.lower() for h in OUTCOME_HINTS)] or 'none'}")
print("  -> no causal question is answerable until a decision log exists. That is the ask.")

THE PRE-FLIGHT CHECK, justified by the measured error pattern

top-50 misses: 5 of 50
 impressions_90d  days_with_impressions  avg_position    content_type trend_direction  trend_pct
            1593                     84           4.1 keyword article              up       23.6
             291                     62           4.4 keyword article              up       93.7
              84                     37           6.2 keyword article          stable       -8.7
             987                     85          10.5 keyword article              up       71.3
            1266                     84           4.6 keyword article              up       84.8
  every miss is a page-1 page: avg_position 4.1-10.5
  measured 'up': 4 of 5
  -> CHECK 1 (confirm the direction on page-1 rows) is not caution, it is where the errors are.

  pages in the top 50 needing the direction check: 38 (76% of the sprint)

CHECK 2 (seasonality) is UNQUANTIFIABLE from this file, stated rather than hidden:


  content types at chance    : ['comparison article'] -> removed from the queue above
  clients below the 200-page floor: 10 clients, 586 pages -> ranking withheld
  base rate travels with every metric: 0.5421 (written into the exported queue's header)

WHAT THE HUMAN MUST SUPPLY - the column this dataset does not have:
  intervention/outcome columns present: none
  -> no causal question is answerable until a decision log exists. That is the ask.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**The premise.** This ranking was fitted on one 90-day window from one portfolio of 32 clients. Everything
below is a way of detecting that the world has moved away from that window. Each trigger has a **measured
value from this build** as its reference point, so "drift" is a comparison rather than a feeling.

### Tier 1 — check every sprint (cheap, from the queue itself)

| Signal | Reference (this build) | Trigger | Why it matters |
|---|---|---|---|
| **Label base rate** | 0.5421 | moves outside 0.45–0.65 | Every precision number is quoted against this. A moved base rate silently re-scales the headline |
| **Realised precision@50** from the decision log | 0.900 (measured proxy) | below **0.70** for two consecutive sprints | 0.70 is the "useful" floor committed to in ML-03 before anything was built |
| **Editor override rate** | n/a — new instrument | >3 overrides for the same reason | The escalation rule in Section 3; this is how `comparison article` got found |
| **Queue concentration** | 58% largest client | any single client >60% after the cap | The cap is doing the work; if it stops, the ranking has drifted toward one portfolio |

### Tier 2 — check quarterly (needs a re-run)

| Signal | Reference | Trigger |
|---|---|---|
| **Per-client AUC spread** | 0.506–0.770, median 0.642 | any client with ≥200 pages falls below 0.55 → withhold that client's queue |
| **Content-type mix** | keyword 90.7%, feedly 7.0%, comparison 2.3% | any type moves >10 points → refit; the model's per-type skill differs enormously |
| **Feature importance profile** | `impressions_90d` first at 0.119 AUC | a different feature takes the top slot → the signal changed, re-read the error analysis |
| **Missingness pattern** | feedly 100% missing `search_volume` | pattern changes → the `has_*` flags now mean something different |

### Tier 3 — immediate refit, no discussion

1. **The label definition changes.** `trend_direction` is a ±20% threshold. **ML-09 found the research
   paper documenting this same field as ±10%** — so this is not hypothetical, the definition is already
   ambiguous across FlyRank's own documents. If the pipeline ever ships ±10%, every number in this project
   describes different cohorts and must be recomputed.
2. **A new client is onboarded.** Measured cost of an unseen client: **0.152 AUC**. Withhold their queue
   until they have history and a measured per-client AUC.
3. **A measurement window changes.** GSC coverage caps at 88 days here and GA4 at 90. If either moves,
   `days_with_impressions` — the rule's 3-point condition — changes meaning.
4. **The upstream population filter changes.** The file is pre-filtered to age ≥ 90 days with traffic > 0.
   Admitting younger or zero-traffic pages changes the population every claim is scoped to.

### What I would instrument first if this went to production

**The decision log, before any of the above.** Every trigger in Tier 1 except the base rate depends on
knowing what the editor actually did. Without it, monitoring is limited to watching the input data drift
and hoping the output still means something — and there would be no way to ever answer whether the refresh
worked. **The cheapest high-value change to this whole project is a three-column CSV: page id, decision,
date.**

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# The monitoring thresholds, computed from THIS build so drift is measurable rather than felt.
monitors = {
    "label_base_rate": {"reference": round(BASE_RATE, 4), "trigger": "outside 0.45-0.65",
                        "tier": 1, "why": "every precision figure is quoted against this"},
    "precision_at_50": {"reference": round(precision_at_k(y, hybrid, 50), 4),
                        "trigger": "realised < 0.70 for two consecutive sprints",
                        "tier": 1, "why": "0.70 is the ML-03 usefulness floor, committed before building"},
    "queue_concentration": {
        "reference": {
            "before_cap": round(float(sprint["client_id"].value_counts().iloc[0] / len(sprint)), 4),
            "after_cap": round(float(sendable["client_id"].value_counts().iloc[0] / len(sendable)), 4),
        },
        "trigger": "largest client > 0.40 of the SENDABLE set (i.e. the cap stopped working)",
        "tier": 1,
        "why": "the raw ranking concentrates at 0.58; the cap is the only thing holding it down"},
    "per_client_auc_min": {"reference": round(float(pc["auc"].min()), 4),
                           "trigger": "any client >=200 pages below 0.55 -> withhold that queue",
                           "tier": 2, "why": "portfolio-wide AUC hides per-client failure"},
    "content_type_mix": {"reference": {k: round(v, 4) for k, v in
                                       df["content_type"].value_counts(normalize=True).round(4).items()},
                         "trigger": "any type moves >10 percentage points", "tier": 2,
                         "why": "per-type skill differs enormously (AUC 0.524 to 0.843)"},
    # NB: the receipt JSON is written with sort_keys=True, so its key ORDER is alphabetical.
    # Take the max by value - reading the first key would report the wrong feature.
    "top_feature": {"reference": max(receipt["top_features_permutation_auc_drop"].items(),
                                     key=lambda kv: kv[1]),
                    "trigger": "a different feature takes the top slot", "tier": 2,
                    "why": "the ranking's basis changed; re-read the error analysis"},
    "label_definition": {"reference": "trend_direction = +/-20% on the 30d impression pair",
                         "trigger": "ANY change to the threshold -> immediate refit", "tier": 3,
                         "why": "ML-09 found the research paper documenting this same field as +/-10%"},
    "new_client": {"reference": "measured cost of an unseen client = 0.152 AUC",
                   "trigger": "any client onboarded -> withhold queue until history exists",
                   "tier": 3, "why": "an unseen client gets the model's weakest output"},
}

print("MONITORING PLAN - reference values from this build\n")
for tier in (1, 2, 3):
    print(f"--- Tier {tier} ---")
    for name, m in monitors.items():
        if m["tier"] == tier:
            print(f"  {name}")
            print(f"    reference : {m['reference']}")
            print(f"    trigger   : {m['trigger']}")
    print()

# The thing that does not exist yet, stated as a requirement rather than a wish.
print("BLOCKED ON: the decision log. Every Tier-1 trigger except the base rate needs it.")
print("  required columns: content_id | decision (refreshed/skipped/monitored) | date | one-line reason")
print(f"  rows it would have covered this sprint: {len(sendable)}")
print("  it is also the ONLY route to a causal claim: without a recorded intervention and a")
print("  post-refresh window, 'refreshing these pages recovers traffic' stays unanswerable.")

MONITORING PLAN - reference values from this build

--- Tier 1 ---
  label_base_rate
    reference : 0.5421
    trigger   : outside 0.45-0.65
  precision_at_50
    reference : 0.9
    trigger   : realised < 0.70 for two consecutive sprints
  queue_concentration
    reference : {'before_cap': 0.58, 'after_cap': 0.2963}
    trigger   : largest client > 0.40 of the SENDABLE set (i.e. the cap stopped working)

--- Tier 2 ---
  per_client_auc_min
    reference : 0.5064
    trigger   : any client >=200 pages below 0.55 -> withhold that queue
  content_type_mix
    reference : {'keyword article': 0.9069, 'feedly article': 0.0699, 'comparison article': 0.0232}
    trigger   : any type moves >10 percentage points
  top_feature
    reference : ('impressions_90d', 0.1185)
    trigger   : a different feature takes the top slot

--- Tier 3 ---
  label_definition
    reference : trend_direction = +/-20% on the 30d impression pair
    trigger   : ANY change to the threshold -> immediate refit
  new_c

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/` — your paper builds on these files.*

Three artifacts, and one deliberate split between them:

| File | Committed? | Why |
|---|---|---|
| `work/outputs/refresh_queue.csv` | **No — gitignored** | 30,000 rows of ranked page data. Datasets never enter git (`work/**/*.csv`), and CI fails any commit that includes one |
| `work/outputs/playbook_metrics.json` | **Yes** | The receipt: queue composition, the operational rules and what they cost, and every monitoring reference value |
| `work/figures/fig4_playbook_funnel.svg` | **Yes** | The figure the paper embeds: what the operational rules do to the sprint queue |

**On the figure.** It shows the funnel from the raw top 50 to the pages actually sent to an editor, with
the measured decline rate at each stage and the base rate drawn as a reference line. It is the honest
picture of a product decision: the rules **remove slots** and only marginally move precision. Anyone
reading it should come away understanding that the filtering is about coverage and safety, **not** about
making the model look better — which is exactly why the raw and filtered numbers are drawn side by side
instead of the filtered one being reported alone.

**The queue's schema**, so the paper can describe it without the file being present:

`queue_rank`, `content_id`, `client_id`, `content_type`, `hybrid_score`, `model_score`, `baseline_score`,
`reason_codes`, `action`, `impressions_90d`, `days_with_impressions`, `avg_position`,
`days_since_last_update`, `content_age_days`, `is_declining_label`.

`is_declining_label` is present **for evaluation only** — it is the proxy outcome, and a production export
would not carry it because it is not knowable at the moment an editor asks for a queue.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

out_dir, fig_dir = Path("work/outputs"), Path("work/figures")
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# --- the full ranked queue (gitignored) ------------------------------------
export = queue.copy()
export["action"] = "not_in_this_sprint"
export.loc[export["queue_rank"] <= CAPACITY_PER_SPRINT, "action"] = sprint["action"].to_numpy()
COLS = ["queue_rank", "content_id", "client_id", "content_type", "hybrid_score", "model_score",
        "baseline_score", "reason_codes", "action", "impressions_90d", "days_with_impressions",
        "avg_position", "days_since_last_update", "content_age_days", "is_declining_label"]
export[COLS].to_csv(out_dir / "refresh_queue.csv", index=False)
print(f"wrote work/outputs/refresh_queue.csv ({len(export):,} rows, gitignored)")

# --- the funnel figure -----------------------------------------------------
SERIES_1, SERIES_2 = "#2a78d6", "#c8622d"
SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e3e2dd"

stages = [
    ("Whole portfolio", df, float(y.mean())),
    (f"Top {CAPACITY_PER_SPRINT} (model rank)", sprint, float(sprint["is_declining_label"].mean())),
    ("After removals", sprint[~sprint["action"].str.startswith("removed")],
     float(sprint[~sprint["action"].str.startswith("removed")]["is_declining_label"].mean())),
    ("Sent to an editor", sendable, float(sendable["is_declining_label"].mean())),
]

fig, ax = plt.subplots(figsize=(7.6, 4.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(GRID)
ax.tick_params(colors=INK_MUTED, labelsize=9, length=0)

names = [s[0] for s in stages]
rates = [s[2] for s in stages]
counts = [len(s[1]) for s in stages]
colors = [GRID] + [SERIES_1] * 3
bars = ax.bar(names, rates, width=0.6, color=colors)
ax.axhline(BASE_RATE, color=INK_MUTED, linewidth=1.4, linestyle=(0, (5, 4)))
ax.annotate(f"base rate {BASE_RATE:.3f}", xy=(3.4, BASE_RATE), xytext=(0, 5),
            textcoords="offset points", ha="right", color=INK_MUTED, fontsize=8.5)
for b, r, c in zip(bars, rates, counts):
    ax.annotate(f"{r:.3f}\nn={c:,}", xy=(b.get_x() + b.get_width() / 2, r), xytext=(0, 6),
                textcoords="offset points", ha="center", color=INK, fontsize=9, fontweight="bold")
ax.annotate("the rules remove SLOTS, not errors -\nthey buy coverage and safety, not precision",
            xy=(2.5, 0.30), color=SERIES_2, fontsize=8.5, ha="center")
ax.set_ylabel("Share measured as declining", color=INK_MUTED, fontsize=9.5)
ax.set_ylim(0, 1.08)
ax.set_title("What the operational rules do to a sprint queue",
             color=INK, fontsize=11.5, loc="left", pad=12)
fig.tight_layout()
fig.savefig(fig_dir / "fig4_playbook_funnel.svg", format="svg", facecolor=SURFACE)
plt.close(fig)
print("wrote work/figures/fig4_playbook_funnel.svg")

print("\nTable view of the figure (never let a chart be the only way to read a number):")
print(pd.DataFrame({"stage": names, "pages": counts, "declining_rate": np.round(rates, 3),
                    "base_rate": round(BASE_RATE, 3)}).to_string(index=False))

# --- the committed receipt -------------------------------------------------
playbook = {
    "base_rate": round(BASE_RATE, 4),
    "capacity_per_sprint": CAPACITY_PER_SPRINT,
    "ranking": receipt["shipped"],
    "ranking_precision_at_50": round(precision_at_k(y, hybrid, 50), 4),
    "operational_rules": {
        "per_client_cap": PER_CLIENT_CAP,
        "min_client_pages": MIN_CLIENT_PAGES,
        "verify_direction_below_position": 11,
        "removed_content_types": at_chance,
    },
    "sprint_composition": {k: int(v) for k, v in sprint["action"].value_counts().items()},
    "sendable": {
        "pages": int(len(sendable)),
        "declining_rate": round(float(sendable["is_declining_label"].mean()), 4),
        "clients": int(sendable["client_id"].nunique()),
        "largest_client_share": round(float(sendable["client_id"].value_counts().iloc[0] / len(sendable)), 4),
        "note": "a product decision layered on the ranking - NOT model performance",
    },
    "monitoring": monitors,
    "blocked_on": "a decision log (content_id | decision | date | reason). Every Tier-1 trigger except "
                  "the base rate needs it, and it is the only route to a causal claim.",
    "random_state": RANDOM_STATE,
    "versions": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
(out_dir / "playbook_metrics.json").write_text(json.dumps(playbook, indent=2, sort_keys=True, default=str))
print("\nwrote work/outputs/playbook_metrics.json (committed - the receipt for this playbook)")

# Public-safety check before anything leaves this notebook.
UNSAFE = ("url", "domain", "title", "query", "keyword_text", "client_name", "slug")
leaky_cols = [c for c in COLS if any(h in c.lower() for h in UNSAFE)]
assert not leaky_cols, f"an identifying column reached the export: {leaky_cols}"
print(f"\npublic-safety check on the export: identifying columns = {leaky_cols or 'none'}")
print("  (content_id / client_id are pseudonyms, and the CSV is gitignored regardless)")

wrote work/outputs/refresh_queue.csv (30,000 rows, gitignored)


wrote work/figures/fig4_playbook_funnel.svg

Table view of the figure (never let a chart be the only way to read a number):
              stage  pages  declining_rate  base_rate
    Whole portfolio  30000           0.542      0.542
Top 50 (model rank)     50           0.900      0.542
     After removals     50           0.900      0.542
  Sent to an editor     27           0.926      0.542

wrote work/outputs/playbook_metrics.json (committed - the receipt for this playbook)

public-safety check on the export: identifying columns = none
  (content_id / client_id are pseudonyms, and the CSV is gitignored regardless)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Every operational rule traces to a measured failure mode, not to a hunch
- [x] The filtered queue's precision is never presented as model performance
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.